# ADEPTRS (IGARSS) Adversarial Pipeline (Sklearn-only)
This notebook implements a **dual-stream** adversarial robustness pipeline aligned with the ADEPTRS/IGARSS framework:

1. **Nominal Specialist (NS)** trained on clean (or poisoned) data  
2. **Robust Guardian (RG)** obtained via **ART `AdversarialTrainer`**  
3. **Consistency Gate (CG)** that fuses NS and RG outputs into: **Nominal**, **AttackFlag**, **Anomaly**

Focus models (no XGBoost):
- Logistic Regression
- SVM (RBF)
- Random Forest
- Neural Net (Sklearn `MLPClassifier`)

Attacks:
- HopSkipJump (HSJ)
- ZOO

> Note: This version is **Sklearn-only** (no PyTorch). PyTorch is only needed if you want a custom neural net; otherwise `MLPClassifier` works fine here.

In [1]:
# If running in a fresh environment, you may need:
# !pip install adversarial-robustness-toolbox scikit-learn numpy pandas

import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import Dict, Any, List, Tuple

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from art.estimators.classification import SklearnClassifier
from art.attacks.evasion import HopSkipJump, ZooAttack
from art.defences.trainer import AdversarialTrainer

C:\Users\Jan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Jan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\art\estimators\certification\__init__.py:30: UserWarning: PyTorch not found. Not importing DeepZ or Interval Bound Propagation functionality
  warnings.warn("PyTorch not found. Not importing DeepZ or Interval Bound Propagation functionality")


## 1) Global configuration
Adjust these once and keep everything else deterministic for repeatability.

In [2]:
SEED = 100
np.random.seed(SEED)

# Update these for your dataset schema
LABEL_COL = "anomaly"          # binary label: 0=nominal, 1=anomaly
DROP_COLS = "channel"        # add non-feature columns here (timestamps, ids, etc.)

TEST_SIZE = 0.05

# Poisoning (train-time only)
POISON_RATES = [0.05, 0.10, 0.20]

# Evasion attacks (test-time only)
EVASION_ATTACKS = ["hsj", "zoo"]

# Consistency Gate thresholds (paper uses TA > TB concept)
TA = 0.70  # nominal specialist threshold
TB = 0.55  # robust guardian threshold
PERMISSIVE_MODE = False

# ART trainer mixing ratio (fraction of adversarial samples used during training)
ADV_TRAIN_RATIO = 0.5

## 2) Data loading and preprocessing

Replace `load_and_prepare()` with your real loader.  
This notebook expects a dataframe where:
- `LABEL_COL` is the binary target
- all other columns (minus DROP_COLS) are numeric features

In [3]:
def load_and_prepare(path_or_df) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    df = pd.read_csv(path_or_df) if isinstance(path_or_df, str) else path_or_df.copy()

    y = df[LABEL_COL].astype(int).values
    X = df.drop(columns=[c for c in df.columns if c in DROP_COLS], errors="ignore").values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
    )
    return X_train, X_test, y_train, y_test

# Example (uncomment and edit):
# X_train, X_test, y_train, y_test = load_and_prepare("your_dataset.csv")

## 3) Label-flip poisoning (train-time)

Applies to training labels only. Test labels remain untouched.

In [4]:
def label_flip_poison(y_train: np.ndarray, rate: float, seed: int = SEED) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    y_poison = y_train.copy()
    n = len(y_poison)
    k = int(np.floor(rate * n))
    idx = rng.choice(n, size=k, replace=False)
    y_poison[idx] = 1 - y_poison[idx]  # binary flip
    return y_poison, idx

## 4) Model builders (Sklearn)

We standardize features for LR/SVM/MLP, not for RF.

In [5]:
def build_logreg() -> Any:
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=800))
    ])

def build_svm() -> Any:
    # probability=True gives predict_proba needed for gate
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", probability=True))
    ])

def build_rf() -> Any:
    return RandomForestClassifier(
        n_estimators=300, random_state=SEED, n_jobs=-1
    )

def build_mlp() -> Any:
    # Sklearn neural net replacement for PyTorch
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="adam",
            max_iter=60,
            random_state=SEED
        ))
    ])

MODEL_FACTORY = {
    "logreg": build_logreg,
    "svm": build_svm,
    "rf": build_rf,
    "mlp": build_mlp,
}

## 5) ART wrappers and attacks

We use **black-box evasion attacks** (HSJ, ZOO), so gradients are not required.  
All models are wrapped as `SklearnClassifier`.

In [6]:
def wrap_sklearn_art(trained_model: Any, clip_values=None) -> SklearnClassifier:
    # clip_values optional; set to (min, max) if you have known feature bounds.
    return SklearnClassifier(model=trained_model, clip_values=clip_values)

def make_attack(art_clf: SklearnClassifier, attack_name: str):
    if attack_name == "hsj":
        # Keep iterations modest for speed; tune later
        return HopSkipJump(classifier=art_clf, max_iter=20, max_eval=1000, init_eval=50)
    if attack_name == "zoo":
        return ZooAttack(classifier=art_clf, max_iter=20, nb_parallel=64)
    raise ValueError(f"Unknown attack: {attack_name}")

## 6) Metrics + utility functions

In [7]:
def predict_score(model: Any, X: np.ndarray) -> np.ndarray:
    # return P(y=1) for binary
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        return proba[:, 1]
    # fallback: decision function -> sigmoid (rare here)
    if hasattr(model, "decision_function"):
        z = model.decision_function(X)
        return 1.0 / (1.0 + np.exp(-z))
    raise ValueError("Model has neither predict_proba nor decision_function")

def metrics_binary(y_true: np.ndarray, y_score: np.ndarray, thresh: float = 0.5) -> Dict[str, float]:
    y_pred = (y_score >= thresh).astype(int)
    out = {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred)),
    }
    # AUC only if both classes present
    if len(np.unique(y_true)) == 2:
        out["auc"] = float(roc_auc_score(y_true, y_score))
    else:
        out["auc"] = float("nan")
    return out

## 7) Consistency Gate

Outputs three states:
- **Nominal**
- **AttackFlag** (NS says nominal, RG says anomaly)
- **Anomaly** (both agree anomaly)

Permissive mode: any alarm => Anomaly.

In [8]:
def consistency_gate(p_anom_nominal: np.ndarray,
                     p_anom_robust: np.ndarray,
                     TA: float,
                     TB: float,
                     permissive: bool = False) -> np.ndarray:
    MA = (p_anom_nominal >= TA).astype(int)
    MB = (p_anom_robust >= TB).astype(int)

    if permissive:
        return np.where((MA == 1) | (MB == 1), "Anomaly", "Nominal")

    out = np.full(len(MA), "Nominal", dtype=object)
    out[(MA == 1) & (MB == 1)] = "Anomaly"
    out[(MA == 0) & (MB == 1)] = "AttackFlag"
    return out

## 8) Robust Guardian training via ART `AdversarialTrainer`

We build attacks on the wrapped classifier and adversarially train on the training split.

In [9]:
def train_robust_guardian(art_clf: SklearnClassifier,
                          X_train: np.ndarray,
                          y_train: np.ndarray,
                          attack_names: List[str],
                          ratio: float = ADV_TRAIN_RATIO) -> SklearnClassifier:
    attacks = [make_attack(art_clf, a) for a in attack_names]
    trainer = AdversarialTrainer(classifier=art_clf, attacks=attacks, ratio=ratio)
    trainer.fit(X_train, y_train)
    return trainer.get_classifier()

## 9) End-to-end experiment runner

For each model and poison rate:
- Train Nominal Specialist (NS)
- Evaluate NS on clean test
- Create adversarial examples on test (HSJ/ZOO) and evaluate NS on attacked test
- Train Robust Guardian (RG) via `AdversarialTrainer`
- Evaluate RG on clean + attacked
- Run Consistency Gate and summarize output distribution

In [10]:
@dataclass
class RunResult:
    model: str
    poison_rate: float
    split: str  # clean / hsj / zoo
    ns_acc: float
    ns_f1: float
    rg_acc: float
    rg_f1: float
    gate_nominal: float
    gate_attackflag: float
    gate_anomaly: float

def run_one_model(X_train, y_train, X_test, y_test, model_name: str, poison_rate: float) -> List[RunResult]:
    # Poison (train labels only)
    if poison_rate > 0:
        y_train_use, poisoned_idx = label_flip_poison(y_train, poison_rate, seed=SEED)
    else:
        y_train_use = y_train.copy()

    # Train Nominal Specialist
    ns = MODEL_FACTORY[model_name]()
    ns.fit(X_train, y_train_use)

    # Wrap as ART
    ns_art = wrap_sklearn_art(ns)

    # Train Robust Guardian (starts from NS weights/fit)
    rg_art = train_robust_guardian(ns_art, X_train, y_train_use, EVASION_ATTACKS, ratio=ADV_TRAIN_RATIO)
    rg = rg_art.model  # underlying sklearn model

    results: List[RunResult] = []

    def eval_split(split_name: str, X_eval: np.ndarray, y_eval: np.ndarray):
        ns_score = predict_score(ns, X_eval)
        rg_score = predict_score(rg, X_eval)

        ns_m = metrics_binary(y_eval, ns_score, thresh=0.5)
        rg_m = metrics_binary(y_eval, rg_score, thresh=0.5)

        gate = consistency_gate(ns_score, rg_score, TA=TA, TB=TB, permissive=PERMISSIVE_MODE)
        gate_nom = float(np.mean(gate == "Nominal"))
        gate_af  = float(np.mean(gate == "AttackFlag"))
        gate_an  = float(np.mean(gate == "Anomaly"))

        results.append(RunResult(
            model=model_name,
            poison_rate=poison_rate,
            split=split_name,
            ns_acc=ns_m["acc"],
            ns_f1=ns_m["f1"],
            rg_acc=rg_m["acc"],
            rg_f1=rg_m["f1"],
            gate_nominal=gate_nom,
            gate_attackflag=gate_af,
            gate_anomaly=gate_an
        ))

    # Clean test
    eval_split("clean", X_test, y_test)

    # Attacked tests
    for a in EVASION_ATTACKS:
        atk = make_attack(ns_art, a)
        X_adv = atk.generate(x=X_test)
        eval_split(a, X_adv, y_test)

    return results

## 10) Run all experiments

Uncomment once you have `X_train, X_test, y_train, y_test`.

In [16]:
# Example:
X_train, X_test, y_train, y_test = load_and_prepare("CSVs\dataset.csv")

all_results = []
for model_name in MODEL_FACTORY.keys():
    for pr in [0.0] + POISON_RATES:
        all_results.extend(run_one_model(X_train, y_train, X_test, y_test, model_name, pr))

results_df = pd.DataFrame([r.__dict__ for r in all_results])
results_df

Adversarial training epochs:   0%|          | 0/20 [00:02<?, ?it/s]


AxisError: axis 1 is out of bounds for array of dimension 1

## 11) Simple reporting helpers

In [11]:
def summarize_results(results_df: pd.DataFrame) -> pd.DataFrame:
    # Aggregate mean over runs (if you later add repetitions)
    cols = ["ns_acc","ns_f1","rg_acc","rg_f1","gate_nominal","gate_attackflag","gate_anomaly"]
    return (results_df
            .groupby(["model","poison_rate","split"], as_index=False)[cols]
            .mean()
            .sort_values(["model","poison_rate","split"]))

# Example:
# summary_df = summarize_results(results_df)
# summary_df